# 05 — Interpretability & Clinical Insights
**Brugada-HUCA ECG Dataset**

This notebook answers: **why does the model flag a patient as Brugada?**

| Method | Model | What it shows |
|---|---|---|
| SHAP | Random Forest | Which features push predictions up/down |
| GradCAM | ResNet1D | Which time points and leads matter most |
| Case studies | ResNet1D | ECG overlaid with saliency for TP / FP / FN |

## 0 — Setup

In [ ]:
import sys, pickle
from pathlib import Path

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'src' / 'config.py').exists()),
    _cwd,
)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import torch

import config as cfg
from preprocessing import load_processed
from dataset import make_loaders
from models import ResNet1D
from train import load_checkpoint
from features import extract_batch

sns.set_theme(style='whitegrid', font_scale=1.1)
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Load data and models
X_train, y_train, ids_train = load_processed('train', cfg.DATA_PROCESSED)
X_test,  y_test,  ids_test  = load_processed('test',  cfg.DATA_PROCESSED)

with open(cfg.MODELS_DIR / 'rf_best.pkl', 'rb') as f:
    rf = pickle.load(f)
with open(cfg.MODELS_DIR / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(cfg.MODELS_DIR / 'test_predictions.pkl', 'rb') as f:
    test_preds = pickle.load(f)

model = ResNet1D(n_leads=cfg.N_LEADS, base_channels=64).to(DEVICE)
load_checkpoint(cfg.MODELS_DIR / 'best_resnet1d.pt', model)
model.eval()

thresh_dl = test_preds['ResNet1D']['threshold']
print('All artifacts loaded.')

---
## Part A — SHAP for Random Forest
### A1 — Compute SHAP Values

In [ ]:
import shap

df_train_feat = pd.read_parquet(cfg.DATA_PROCESSED / 'features_train.parquet')
df_test_feat  = pd.read_parquet(cfg.DATA_PROCESSED / 'features_test.parquet')

X_tr_ml = scaler.transform(df_train_feat.values)
X_te_ml = scaler.transform(df_test_feat.values)

# Use the training set as background (TreeExplainer is exact)
explainer = shap.TreeExplainer(rf, data=X_tr_ml, feature_perturbation='interventional')

# Compute SHAP values for Brugada class (class 1)
shap_vals = explainer.shap_values(X_te_ml)

# Handle both old SHAP (list of arrays per class) and new SHAP (3-D array)
if isinstance(shap_vals, list):
    sv = shap_vals[1]
elif shap_vals.ndim == 3:
    sv = shap_vals[:, :, 1]
else:
    sv = shap_vals

print(f'SHAP values shape: {sv.shape}  (test samples × features)')

### A2 — Global Feature Importance (SHAP summary)

In [ ]:
feat_names = df_test_feat.columns.tolist()

# Ensure sv is 2-D (n_samples, n_features) — guard against 3-D SHAP output
if sv.ndim == 3:
    sv = sv[:, :, 1] if sv.shape[2] > 1 else sv[:, :, 0]

# Mean |SHAP| per feature, top 20
mean_shap  = np.abs(sv).mean(axis=0)
top20_idx  = np.argsort(mean_shap)[-20:].tolist()   # plain Python ints
top20_vals = mean_shap[top20_idx]
top20_names = [feat_names[i] for i in top20_idx]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_names, top20_vals, color='steelblue', edgecolor='black')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 20 Features — SHAP Global Importance (Random Forest)')

# Highlight Brugada-specific features
for label in ax.get_yticklabels():
    if any(lead.lower() in label.get_text() for lead in ['v1', 'v2', 'v3']):
        label.set_color('darkorange')
        label.set_fontweight('bold')

ax.text(0.98, 0.02, 'Orange = V1/V2/V3 features', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=9, color='darkorange')

plt.tight_layout()
plt.savefig(REPORTS / 'shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### A3 — SHAP Beeswarm (value × impact)

In [ ]:
ev = explainer.expected_value
base_val = (ev[1] if isinstance(ev, (list, np.ndarray)) and len(np.atleast_1d(ev)) > 1
            else float(np.atleast_1d(ev)[0]))

shap_exp = shap.Explanation(
    values      = sv[:, top20_idx],
    base_values = base_val,
    data        = X_te_ml[:, top20_idx],
    feature_names = [feat_names[i] for i in top20_idx],
)

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_exp, max_display=20, show=False)
plt.title('SHAP Beeswarm — Top 20 Features (Brugada class)')
plt.tight_layout()
plt.savefig(REPORTS / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

### A4 — Individual SHAP Waterfall (one Brugada patient)

In [ ]:
# Pick a true-positive Brugada patient
dl_prob_test = test_preds['ResNet1D']['y_prob']
tp_mask = (y_test == 1) & (dl_prob_test >= thresh_dl)
tp_idx  = np.where(tp_mask)[0][0]

ev = explainer.expected_value
base_val = (ev[1] if isinstance(ev, (list, np.ndarray)) and len(np.atleast_1d(ev)) > 1
            else float(np.atleast_1d(ev)[0]))

patient_exp = shap.Explanation(
    values       = sv[tp_idx, top20_idx],
    base_values  = base_val,
    data         = X_te_ml[tp_idx, top20_idx],
    feature_names = [feat_names[i] for i in top20_idx],
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(patient_exp, show=False)
plt.title(f'SHAP Waterfall — Brugada Patient ID: {ids_test[tp_idx]} (True Positive)')
plt.tight_layout()
plt.savefig(REPORTS / 'shap_waterfall_tp.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part B — GradCAM for ResNet1D

GradCAM produces a 1-D saliency map over time.  
Upsampled to 1200 samples, it can be overlaid directly on the ECG trace.

### B1 — GradCAM Implementation

In [ ]:
from scipy.ndimage import zoom as ndimage_zoom

def gradcam_1d(model, signal_np: np.ndarray, device: str = 'cpu') -> np.ndarray:
    """
    Compute GradCAM saliency map for a single ECG signal.

    Parameters
    ----------
    signal_np : (12, 1200) preprocessed signal

    Returns
    -------
    saliency : (1200,) float array in [0, 1]
    """
    model.eval()
    x = torch.from_numpy(signal_np[np.newaxis]).float().to(device)  # (1, 12, 1200)

    activations, gradients = [], []

    def fwd_hook(module, inp, out):
        activations.append(out.detach())

    target = model.layer4[-1].conv2
    h_fwd  = target.register_forward_hook(fwd_hook)
    h_bwd  = target.register_full_backward_hook(
        lambda m, gi, go: gradients.append(go[0].detach())
    )

    # Forward + backward
    logit = model(x)
    model.zero_grad()
    logit[0, 0].backward()

    h_fwd.remove()
    h_bwd.remove()

    # Compute CAM
    act  = activations[0].squeeze(0)          # (C, L)
    grad = gradients[0].squeeze(0)            # (C, L)

    weights = grad.mean(dim=-1, keepdim=True) # (C, 1)
    cam     = torch.relu((weights * act).sum(dim=0)).cpu().numpy()  # (L,)

    # Normalise & upsample to original signal length
    cam -= cam.min()
    cam /= (cam.max() + 1e-8)
    cam  = ndimage_zoom(cam, cfg.N_SAMPLES / len(cam))
    return cam.astype(np.float32)

print('GradCAM function defined.')

### B2 — Helper: ECG + GradCAM overlay

In [ ]:
def plot_ecg_gradcam(signal, saliency, title, leads_to_show=None, save_path=None):
    """
    Plot ECG leads with GradCAM saliency as a colour overlay.

    leads_to_show : list of lead names; defaults to V1/V2/V3
    """
    if leads_to_show is None:
        leads_to_show = cfg.BRUGADA_LEADS

    n    = len(leads_to_show)
    time = np.arange(cfg.N_SAMPLES) / cfg.FS
    cmap = plt.get_cmap('hot_r')

    fig, axes = plt.subplots(n, 1, figsize=(16, 3 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, lead_name in zip(axes, leads_to_show):
        li  = cfg.LEAD_NAMES.index(lead_name)
        sig = signal[li]

        y_lo = float(sig.min()) - 0.3
        y_hi = float(sig.max()) + 0.3

        # imshow for saliency overlay — no dimension constraints
        ax.imshow(
            saliency[np.newaxis, :],
            aspect='auto',
            extent=[time[0], time[-1], y_lo, y_hi],
            cmap=cmap, vmin=0, vmax=1,
            alpha=0.55, origin='lower', zorder=2,
        )

        ax.plot(time, sig, color='black', linewidth=1.0, zorder=3)
        ax.set_ylabel(lead_name, fontweight='bold')
        ax.set_ylim(y_lo, y_hi)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title, fontsize=13, fontweight='bold')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    fig.colorbar(sm, ax=axes, label='GradCAM Saliency', shrink=0.6, pad=0.02)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print('Plot function defined.')

### B3 — Case Study 1: True Positive Brugada

In [ ]:
# True Positive: Brugada correctly detected
tp_idx = np.where((y_test == 1) & (dl_prob_test >= thresh_dl))[0][0]
tp_sig = X_test[tp_idx]
tp_cam = gradcam_1d(model, tp_sig, device=DEVICE)
tp_prob = dl_prob_test[tp_idx]

plot_ecg_gradcam(
    tp_sig, tp_cam,
    title  = f'TRUE POSITIVE — Brugada ID: {ids_test[tp_idx]} | P(Brugada)={tp_prob:.3f}',
    save_path = REPORTS / 'gradcam_true_positive.png',
)

### B4 — Case Study 2: True Negative Normal

In [ ]:
tn_idx = np.where((y_test == 0) & (dl_prob_test < thresh_dl))[0][0]
tn_sig = X_test[tn_idx]
tn_cam = gradcam_1d(model, tn_sig, device=DEVICE)
tn_prob = dl_prob_test[tn_idx]

plot_ecg_gradcam(
    tn_sig, tn_cam,
    title  = f'TRUE NEGATIVE — Normal ID: {ids_test[tn_idx]} | P(Brugada)={tn_prob:.3f}',
    save_path = REPORTS / 'gradcam_true_negative.png',
)

### B5 — Case Study 3: False Positive (if any)

In [ ]:
fp_mask = (y_test == 0) & (dl_prob_test >= thresh_dl)

if fp_mask.any():
    fp_idx = np.where(fp_mask)[0][0]
    fp_sig  = X_test[fp_idx]
    fp_cam  = gradcam_1d(model, fp_sig, device=DEVICE)
    fp_prob = dl_prob_test[fp_idx]

    plot_ecg_gradcam(
        fp_sig, fp_cam,
        title     = f'FALSE POSITIVE — Normal ID: {ids_test[fp_idx]} | P(Brugada)={fp_prob:.3f}',
        save_path = REPORTS / 'gradcam_false_positive.png',
    )
else:
    print('No false positives in test set.')

---
## Part C — Lead Importance from GradCAM

By computing GradCAM over all test samples and aggregating **per-lead signal energy**,  
we can quantify which leads the model relied on most — and verify it aligns with clinical knowledge (V1/V2/V3).

In [ ]:
def lead_saliency_scores(model, X: np.ndarray, device: str = 'cpu') -> np.ndarray:
    """
    For each sample, compute GradCAM saliency and weight each lead
    by its mean absolute signal * saliency product.

    Returns
    -------
    scores : (N, 12) — higher = more important
    """
    scores = []
    for i in range(len(X)):
        cam = gradcam_1d(model, X[i], device=device)   # (1200,)
        lead_scores = np.array([
            float((np.abs(X[i, li]) * cam).mean())
            for li in range(cfg.N_LEADS)
        ])
        scores.append(lead_scores)
    return np.array(scores)

# Run on Brugada test samples only (to see what drives the Brugada prediction)
brugada_test_idx = np.where(y_test == 1)[0]
X_b_test = X_test[brugada_test_idx]

print(f'Computing lead saliency for {len(X_b_test)} Brugada test samples...')
lead_scores_b = lead_saliency_scores(model, X_b_test, device=DEVICE)

# Mean per lead
mean_lead_scores = lead_scores_b.mean(axis=0)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#C62828' if ln in cfg.BRUGADA_LEADS else '#1565C0' for ln in cfg.LEAD_NAMES]
ax.bar(cfg.LEAD_NAMES, mean_lead_scores, color=colors, edgecolor='black')
ax.set_ylabel('Mean Saliency Score')
ax.set_title('Lead Importance via GradCAM — Brugada Test Samples')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#C62828', label='Brugada-specific (V1/V2/V3)'),
    Patch(color='#1565C0', label='Other leads'),
])
plt.tight_layout()
plt.savefig(REPORTS / 'gradcam_lead_importance.png', dpi=150)
plt.show()

---
## Summary & Clinical Takeaways

In [ ]:
top_lead = cfg.LEAD_NAMES[int(mean_lead_scores.argmax())]
v1_rank  = int(np.argsort(-mean_lead_scores).tolist().index(cfg.LEAD_NAMES.index('V1'))) + 1
v2_rank  = int(np.argsort(-mean_lead_scores).tolist().index(cfg.LEAD_NAMES.index('V2'))) + 1
v3_rank  = int(np.argsort(-mean_lead_scores).tolist().index(cfg.LEAD_NAMES.index('V3'))) + 1

print('=== Clinical Interpretability Summary ===')
print(f'Most important lead (GradCAM):  {top_lead}')
print(f'V1 rank: {v1_rank} / {cfg.N_LEADS}')
print(f'V2 rank: {v2_rank} / {cfg.N_LEADS}')
print(f'V3 rank: {v3_rank} / {cfg.N_LEADS}')
print()
print('SHAP top features should highlight V1/V2/V3 ST-elevation and J-point features.')
print('If V1/V2/V3 dominate both SHAP and GradCAM, the model has learned')
print('the clinically correct coved ST-elevation pattern.')
print()
print('Figures saved to reports/:')
for f in sorted(REPORTS.glob('*.png')):
    print(f'  {f.name}')